# Evaluation 07 — Statistical analysis across training seeds

This notebook reads the actual verified Evaluation 04 reports. It produces **mean ± sample standard deviation** of held-out real-test macro-F1, balanced accuracy, and accuracy across trained seeds; per-class recall/precision/F1 across seeds; and paired E0–E5 real-test differences across matching seeds. It does not create or infer any results. **Bootstrap 95% intervals come from Evaluation 04** using frozen protected real-image groups; the same real patients are reused across seeds, so these summaries are not additional independent patients or new confidence intervals. Avoid p-values or claims of statistical significance without a prespecified valid test.

In [ ]:
from pathlib import Path
import sys,subprocess,json
REPO_ROOT=Path('/kaggle/working/DermSynReal')
if not (REPO_ROOT/'src/dermsynreal/statistical_analysis.py').is_file():
    subprocess.check_call(['git','clone','--depth','1',
        'https://github.com/shanmukhdatta/DermSynReal.git',str(REPO_ROOT)])
sys.path.insert(0,str(REPO_ROOT/'src'))
from dermsynreal.kaggle_runs import start_notebook_logging,append_run_log
from dermsynreal.statistical_analysis import summarize_across_seeds


## Attach Evaluation 04 output and select a new output location

Expected: `run_summary.csv`, `per_class_real_test.csv`, `paired_real_test.csv`. Run Evaluation 04 after all trained model checkpoints were evaluated on the exact same real test split.

In [ ]:
RESULTS_DIR="/kaggle/input/EDIT_EVAL_04_RESULTS"
OUTPUT_DIR="/kaggle/working/dermsynreal_eval_07"
RUN_STATISTICAL_ANALYSIS=False
start_notebook_logging(OUTPUT_DIR,kind="evaluation_statistical_analysis",config={
    "input_set":"Evaluation 04 actual measured CSVs","results_dir":"private Kaggle mount"
},repo_root=REPO_ROOT)


In [ ]:
if RUN_STATISTICAL_ANALYSIS:
    import pandas as pd
    summary=summarize_across_seeds(RESULTS_DIR,OUTPUT_DIR)
    print(json.dumps(summary,indent=2))
    for file in ("seed_summary_real_test.csv","seed_summary_per_class.csv",
                 "seed_summary_paired_contrasts.csv"):
        frame=pd.read_csv(Path(OUTPUT_DIR)/file)
        print(file)
        display(frame)
    append_run_log(OUTPUT_DIR,kind="evaluation_statistical_analysis_finished",
        config={"result":"created only from real Evaluation 04 CSVs",
                "real_test_n":summary["real_test_n"]},
        artifacts=[Path(OUTPUT_DIR)/"seed_summary_real_test.csv",
                   Path(OUTPUT_DIR)/"seed_summary_per_class.csv",
                   Path(OUTPUT_DIR)/"seed_summary_paired_contrasts.csv",
                   Path(OUTPUT_DIR)/"statistical_analysis_summary.json"],
        repo_root=REPO_ROOT)
else:
    print("Statistical analysis locked: attach actual Evaluation 04 output first.")


In [ ]:
if RUN_STATISTICAL_ANALYSIS:
    import pandas as pd
    import matplotlib.pyplot as plt
    table=pd.read_csv(Path(OUTPUT_DIR)/"seed_summary_real_test.csv")
    subset=table[table["metric"]=="real_test_macro_f1"].copy()
    if not subset.empty:
        subset["name"]=subset["backbone"]+" / "+subset["condition"]
        yerr=subset["sample_sd"].fillna(0).astype(float)
        fig,ax=plt.subplots(figsize=(max(8,len(subset)*0.8),5))
        ax.bar(subset["name"],subset["mean"].astype(float),yerr=yerr,capsize=3)
        ax.set_ylim(0,1)
        ax.set_ylabel("Real-test macro-F1, mean ± training-seed SD")
        ax.set_title("Descriptive across-seed scores (same real test)")
        ax.tick_params(axis="x",rotation=60)
        fig.tight_layout()
        fig.savefig(Path(OUTPUT_DIR)/"real_test_macro_f1_across_seeds.png",dpi=180)
        plt.show()


## Publication use

Copy the verified *aggregate* tables into the manuscript after checking source labels, leakage, real held-out IDs, missing arms, group independence and model checkpoint identities. Do not treat a single seed's SD as zero, claim that repeated training seeds create a larger test cohort, or report a bootstrap CI as a p-value. The Kaggle folder contains private `notebook_console.log` and `run_log.jsonl`; a separate safe aggregate-export script prepares selected CSVs for GitHub.